# 🚗 YOLOv8 Autonomous Vehicle Detection - Training
This notebook is designed to train a YOLOv8 model on a custom dataset for autonomous vehicle detection. It uses the **YOLOv8 Small (s)** model for a balance between speed and accuracy.

### Steps:
1. Setup Environment.
2. Unzip & Prepare Data.
3. Load Pre-trained Model.
4. Train the Model.
5. Evaluate Results.
6. Export & Download.

# 0. Import Libraries
Import necessary libraries for file handling, image processing (OpenCV), and augmentation (Albumentations).

In [1]:
import kagglehub
import glob
import os
import shutil
import zipfile
from IPython.display import Image, display

# 1. Install ultralytics 
Set up the Ultralytics Library 

In [2]:
#  Setup ultralytics Environment
target_file = "/kaggle/input/ultralytics/ultralytics-8.3.199-py3-none-any.whl"
os.system(f"pip install '{target_file}' --no-deps")

import ultralytics
from ultralytics import YOLO
# Check setup
print(f"Ultralytics version: {ultralytics.__version__}")

Processing /kaggle/input/ultralytics/ultralytics-8.3.199-py3-none-any.whl
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.3.199


## 2. Prepare Dataset
We will extract the processed dataset (`final_dataset.zip`) generated from the preprocessing stage.

In [3]:
# --- 1. Setup Paths ---
# Update this path to match your input dataset
input_folder_path = "/kaggle/input/ready-datasetv2/Ready_Dataset" 
destination_path = "/kaggle/working/Ready_Dataset"

# --- 2. Reset & Copy Dataset ---
# Remove existing folder to avoid "FileExistsError"
if os.path.exists(destination_path):
    shutil.rmtree(destination_path)

print(f"📂 Copying dataset to {destination_path}...")

# Copy directly (No try/except - will fail loudly if something is wrong)
shutil.copytree(input_folder_path, destination_path)
print("✅ Dataset successfully copied!")

# --- 3. Verify Config (Final Check) ---
yaml_path = os.path.join(destination_path, "data.yaml")

if os.path.exists(yaml_path):
    print(f"📄 Data configuration found: {yaml_path}")
else:
    print("⚠️ Warning: 'data.yaml' not found. Check your dataset structure.")

📂 Copying dataset to /kaggle/working/Ready_Dataset...
✅ Dataset successfully copied!
📄 Data configuration found: /kaggle/working/Ready_Dataset/data.yaml


## 3. Load Model
We initialize the **YOLOv8 Small (s)** model.
* **Nano (n):** Fastest, lowest accuracy.
* **Small (s):** Balanced, better for small objects (traffic lights, pedestrians).

In [5]:
# 3. Load YOLOv8 Small Model
# Using 'yolov8s.pt' for better detection accuracy in autonomous driving scenarios
model = YOLO('/kaggle/input/yolov8s/yolov8s.pt') 

print("🤖 Model loaded: YOLOv8s (Small)")

🤖 Model loaded: YOLOv8s (Small)


## 4. Start Training 🚀
We will train the model with the following hyperparameters:
* **Epochs:** 50 (Sufficient for initial convergence).
* **Image Size:** 416 (Matches our preprocessed data).
* **Batch Size:** 32 (Optimized for T4 GPU memory).
* **Patience:** 15 (Early stopping if no improvement).

In [ ]:
# 4. Train the Model
print("🚀 Starting Training (Cleaned Data)...")

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=416,
    batch=256,
    patience=10,
    name='yolov8s_autonomous',
    device=[1,0],
    amp=True,   
    workers=4,    
    save=True,
    exist_ok=True,
    verbose=True
)

🚀 Starting Training (Cleaned Data)...
New https://pypi.org/project/ultralytics/8.3.233 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.199 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:1 (Tesla T4, 15095MiB)
                                                        CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=256, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Ready_Dataset/data.yaml, degrees=0.0, deterministic=True, device=1,0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=30

## 5. Evaluation & Visualization
Let's look at the training curves (Loss, mAP) and the Confusion Matrix to understand model performance.

In [ ]:
# 5. Visualization
run_path = "/kaggle/working/runs/detect/yolov8s_autonomous"

# Display Learning Curves
results_img = os.path.join(run_path, "results.png")
if os.path.exists(results_img):
    print("📊 Training Metrics (Loss & mAP):")
    display(Image(filename=results_img, width=800))
else:
    print("⚠️ Results image not found.")

# Display Confusion Matrix
conf_matrix = os.path.join(run_path, "confusion_matrix.png")
if os.path.exists(conf_matrix):
    print("\n🧩 Confusion Matrix:")
    display(Image(filename=conf_matrix, width=600))

# Display Learning Curves
result_img = "/kaggle/working/runs/detect/val/BoxF1_curve.png"
if os.path.exists(result_img):
    print("📊 Training Metrics (Loss & mAP):")
    display(Image(filename=result_img, width=800))
else:
    print("⚠️ Results image not found.")

## 6. Final Validation & Download
Run a final check on the **Test Set** and generate a download link for the trained weights (`best.pt`).

In [ ]:
# 6. Validation & Download
print("🔍 Running Final Validation on Test Set...")

# Load the best trained weights
best_weight_path = os.path.join(run_path, "weights/best.pt")
if os.path.exists(best_weight_path):
    best_model = YOLO(best_weight_path)

    # Run validation on 'test' split
    metrics = best_model.val(split='test')
    
    print(f"\n🎯 Final mAP@50: {metrics.box.map50}")
    print(f"🎯 Final mAP@50-95: {metrics.box.map}")

    # Create Download Link
    from IPython.display import FileLink
    print("\n💾 Download Best Model:")
    display(FileLink(best_weight_path))
else:
    print("❌ Best weights not found. Training might have failed.")